# 🚀 THSA-2B: Real Pretrained Foundation SFT + LoRA Training (Google Colab)
### Fine-Tuning Multilingual Foundation LLM on 5-Tier Bilingual ShareGPT Curriculum

This notebook fine-tunes a pretrained multilingual foundation model (`Qwen/Qwen2.5-0.5B-Instruct` with full native Bengali and English vocabulary) on our **5-Tier Progressive ShareGPT Dataset** (Micro-Greetings, Stress Management, Philosophy, 1971 Liberation War, and STEM Physics/Math).

**Training Time on Colab T4 GPU:** ~3 to 5 minutes.

In [ ]:
# Step 1: Check GPU Acceleration
!nvidia-smi

In [ ]:
# Step 2: Clone the Official Repository
!git clone https://github.com/tpxplatfrom-afk/SS_module_BD.git
%cd SS_module_BD/ss_bangladesh_nano_android_module/THSA-2B\ V1

In [ ]:
# Step 3: Install Required Dependencies
!pip install -q torch transformers datasets peft accelerate sentencepiece bitsandbytes trl

In [ ]:
# Step 4: Generate 5-Tier Progressive ShareGPT Dataset
!python data/build_bilingual_sharegpt_dataset.py

## 🏋️ Step 5: Run Foundation Model SFT + LoRA Fine-Tuning
Fine-tunes the pretrained base weights on our 5-tier curriculum and merges the LoRA adapters into standalone self-contained weights.

In [ ]:
!python training/train_foundation_sft.py \
    --base_model Qwen/Qwen2.5-0.5B-Instruct \
    --train_data data/train_sharegpt.jsonl \
    --test_data data/test_sharegpt.jsonl \
    --output_dir checkpoints/thsa_foundation_merged \
    --epochs 5 \
    --batch_size 2 \
    --lr 2e-4 \
    --lora_r 16 \
    --lora_alpha 32

## 🧪 Step 6: Live Neural Generation & Quality Verification Test in Colab
Tests interactive text generation directly from the fine-tuned model to verify novel sentence generation in Bangla & English.

In [ ]:
!python tools/export_foundation_to_nano.py \
    --model_dir checkpoints/thsa_foundation_merged

## 💬 Step 7: Interactive Chat directly inside Colab

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('checkpoints/thsa_foundation_merged')
model = AutoModelForCausalLM.from_pretrained(
    'checkpoints/thsa_foundation_merged',
    torch_dtype=torch.float16,
    device_map='auto'
)

def chat_with_shanto(user_prompt):
    messages = [{'role': 'user', 'content': user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.9, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print('🤖 Shanto AI:\n' + response.strip())

# Test your own custom prompt here:
chat_with_shanto('হাই শান্ত! কেমন আছো? ১৯৭১ সালের মুক্তিযুদ্ধ সম্পর্কে কিছু বলো।')

## 📦 Step 8: Download Fine-Tuned Model Weights & Tokenizer

In [ ]:
!zip -r thsa_foundation_merged.zip checkpoints/thsa_foundation_merged
from google.colab import files
files.download('thsa_foundation_merged.zip')